In [1]:
from typing import cast
import pandas as pd
import numpy as np
import bs4
import requests
from typing import Coroutine
import asyncio
import httpx
from typing import Any
SEMAFORO=asyncio.Semaphore(20)

In [2]:
url_principal:str='https://www.pisos.com'
soup:bs4.BeautifulSoup=bs4.BeautifulSoup(requests.get(url=url_principal+'/viviendas/madrid').text,'html.parser')
links:bs4.ResultSet=cast(bs4.Tag,soup.find(class_='zoneList')).find_all(class_='seo-box__location-link--level2')
zonas_madrid:list[str]=[link['href'] for link in links]
zonas_madrid

['/viviendas/corredor_del_henares/',
 '/viviendas/madrid_capital/',
 '/viviendas/madrid_noroeste/',
 '/viviendas/madrid_norte/',
 '/viviendas/madrid_sur/',
 '/viviendas/madrid_sureste/',
 '/viviendas/madrid_suroeste/']

In [3]:
async def consultar_html_region(url:str,client:httpx.AsyncClient)->str:
    result:httpx.Response=await client.get(url_principal+url)
    return result.text
async def consultar_htmls_zonas(zonas:list[str])->list[str]:
    async with httpx.AsyncClient() as client:
        tareas:list[Coroutine[Any,Any,str]]=[consultar_html_region(zona,client) for zona in zonas]
        tareas_completas:list[str]=await asyncio.gather(*tareas)
        return tareas_completas
htmls_zonas:list[str]=await consultar_htmls_zonas(zonas_madrid)
transform_zonas:list[bs4.Tag]=[cast(bs4.Tag,bs4.BeautifulSoup(html,'html.parser').body) for html in htmls_zonas]

In [4]:
def sacar_n_boton(zona:bs4.Tag)->tuple[str,int]:
    boton:bs4.Tag=cast(bs4.Tag,zona.find(class_='button__primary'))
    return str(boton['href']),int(boton.text.split(' ')[1].replace('.',''))
async def consultar_urls_zonas(zona:bs4.Tag)->list[str]:
    lista:list[str]=[]
    en,n_resultados=sacar_n_boton(zona)
    if n_resultados>3000:
        zoneList:bs4.Tag=cast(bs4.Tag,zona.find(class_='zoneList'))
        enlaces:list[bs4.Tag]=zoneList.find_all(class_='seo-box__location-link--level2')
        hrefs:list[str]=[str(a['href']) for a in enlaces]
        hrefs_viviendas:list[str]=list(filter(lambda x:'/viviendas/' in x,hrefs))
        hrefs_ventas:list[str]=list(filter(lambda x:x not in hrefs_viviendas,hrefs))
        htmls:list[str]=await consultar_htmls_zonas(hrefs_viviendas)
        transforms:list[bs4.Tag]=[cast(bs4.Tag,bs4.BeautifulSoup(html,'html.parser').body) for html in htmls]
        lista.extend(href for href,n in [sacar_n_boton(z) for z in transforms])
        lista.extend(hrefs_ventas)
    else:
        lista=[en]
    return lista
lista_urls_zonas:list[str]=[url for zona in [(await consultar_urls_zonas(zona)) for zona in transform_zonas] for url in zona]
lista_urls_zonas

['/venta/pisos-corredor_del_henares/',
 '/venta/pisos-arganzuela/',
 '/venta/pisos-madrid_capital_salamanca/',
 '/venta/pisos-madrid_capital_carabanchel/',
 '/venta/pisos-madrid_capital_centro/',
 '/venta/pisos-madrid_capital_chamartin/',
 '/venta/pisos-chamberi_distrito/',
 '/venta/pisos-ciudad_lineal/',
 '/venta/pisos-fuencarral_el_pardo/',
 '/venta/pisos-hortaleza/',
 '/venta/pisos-latina/',
 '/venta/pisos-moncloa_aravaca/',
 '/venta/pisos-puente_de_vallecas/',
 '/venta/pisos-madrid_capital_retiro/',
 '/venta/pisos-madrid_capital_san_blas/',
 '/venta/pisos-tetuan/',
 '/venta/pisos-madrid_capital_usera/',
 '/venta/pisos-madrid_capital_vicalvaro/',
 '/venta/pisos-villa_de_vallecas/',
 '/venta/pisos-villaverde_distrito/',
 '/venta/pisos-madrid_capital_barajas/',
 '/venta/pisos-moratalaz/',
 '/venta/pisos-madrid_noroeste/',
 '/venta/pisos-madrid_norte/',
 '/venta/pisos-madrid_sur/',
 '/venta/pisos-madrid_sureste/',
 '/venta/pisos-madrid_suroeste/']

In [5]:
async def consultar_n_paginas_opcion_localidad(id:int,u:str,client:httpx.AsyncClient)-> dict[int,int]:
    result:httpx.Response=await client.get(url_principal+u)
    posible:bs4.Tag | None=bs4.BeautifulSoup(result.text,'html.parser').find(class_='grid__title')
    n:str='0'
    if title:=posible:
        r:str=title.find_all('span')[1].text
        if len(r)>0:
            n=r.split(' ')[0]
    resultados:int=int(n.replace('.',''))
    n_paginas_completas:int=resultados//30
    n_paginas:int=int(np.min([n_paginas_completas+(resultados>(30*n_paginas_completas)),100]))
    return {id:n_paginas}
async def sacar_n_paginas()->dict[int,int]:
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1)) as client:
        tareas:list[Coroutine[Any,Any,dict[int,int]]]=[consultar_n_paginas_opcion_localidad(id,u,client) for id,u in enumerate(lista_urls_zonas)]
        tareas_completas:list[dict[int,int]]=await asyncio.gather(*tareas)
        return {k:v for d in tareas_completas for k,v in d.items()}
n_paginas:dict[int,int]=await sacar_n_paginas()
n_paginas

{0: 31,
 1: 10,
 2: 46,
 3: 13,
 4: 33,
 5: 12,
 6: 20,
 7: 15,
 8: 8,
 9: 8,
 10: 10,
 11: 11,
 12: 13,
 13: 13,
 14: 10,
 15: 11,
 16: 8,
 17: 5,
 18: 6,
 19: 9,
 20: 3,
 21: 3,
 22: 59,
 23: 32,
 24: 51,
 25: 13,
 26: 25}

In [6]:
async def consultar_anuncios_cargados(client:httpx.AsyncClient,url:str,pagina:int)->str:# junta todos los anuncios de una página de una localidad
    async with SEMAFORO:
        result:httpx.Response=await client.get(f"{url_principal}{url}{pagina}",timeout=10)
        return result.text
async def consultar_anuncios_url(client:httpx.AsyncClient,url:str)->list[str]: # junta todos los anuncios de una localidad
    tareas:list[Coroutine[Any,Any,str]]=[consultar_anuncios_cargados(client,url,i) for i in range(1,n_paginas[lista_urls_zonas.index(url)]+1)]
    tareas_completas:list[str]=await asyncio.gather(*tareas)
    return tareas_completas
async def consultar_anuncios(lista_urls:list[str])->list[str]: # junta todos los anuncios
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1),limits=httpx.Limits(max_connections=20,max_keepalive_connections=20)) as client:
        tareas:list[Coroutine[Any,Any,list[str]]]=[consultar_anuncios_url(client,url) for url in lista_urls]
        tareas_completas:list[list[str]]=await asyncio.gather(*tareas)
        anuncios:list[str]=[u for lista in tareas_completas for u in lista]
    transformados:list[bs4.ResultSet]=[bs4.BeautifulSoup(ad,'html.parser').find_all(class_='ad-preview') for ad in anuncios]
    lista_ads=[ad for rs in transformados for ad in rs]
    return [f"{url_principal}{ad['data-lnk-href']}" for ad in lista_ads]

In [ ]:
ventas_ads:list[str]=list(set(await consultar_anuncios(lista_urls_zonas)))
ventas_ads

['https://www.pisos.com/comprar/piso-zona_sureste-63411718761_100500/',
 'https://www.pisos.com/comprar/piso-pryconsa_juan_de_austria28804-64242618195_994867/',
 'https://www.pisos.com/comprar/piso-pryconsa_juan_de_austria28804-65043163226_271200/',
 'https://www.pisos.com/comprar/piso-rinconada28802-64198451305_271200/',
 'https://www.pisos.com/comprar/casa_adosada-loeches_centro_urbano-65896755914_106100/',
 'https://www.pisos.com/comprar/piso-rinconada28803-65028271140_271200/',
 'https://www.pisos.com/comprar/piso-zona_noroeste-65929071312_101000/',
 'https://www.pisos.com/comprar/chalet_adosado-zona_suroeste-54185554852_517810/',
 'https://www.pisos.com/comprar/duplex-avicola_san_nicolas-63407639828_100500/',
 'https://www.pisos.com/comprar/piso-zona_sureste-63396336971_100500/',
 'https://www.pisos.com/comprar/piso-pryconsa_juan_de_austria28804-64205427845_100200/',
 'https://www.pisos.com/comprar/piso-coslada_casco_antiguo-64241709886_100200/',
 'https://www.pisos.com/comprar/ca

In [17]:
ventas_ads=list(set(ventas_ads))

In [18]:
with open('./anuncios.txt','w') as f:
    f.writelines([f"{ad}\n" for ad in ventas_ads])

In [19]:
with open('./anuncios.txt','r') as f:
    ads_copia=[ad[:-2] for ad in f.readlines()]
ads_copia

['https://www.pisos.com/comprar/atico-cortes_huertas28014-63393178826_109200',
 'https://www.pisos.com/comprar/piso-justicia_chueca28004-62508498356_109200',
 'https://www.pisos.com/comprar/piso-madrid_capital_carabanchel-64252985931_106700',
 'https://www.pisos.com/comprar/piso-fuente_del_berro28017-63384011882_102100',
 'https://www.pisos.com/comprar/piso-ciudad_lineal_colina-63391210948_109200',
 'https://www.pisos.com/comprar/piso-latina_aluche28024-63375110020_518874',
 'https://www.pisos.com/comprar/piso-alcobendas_centro28100-65045189989_109800',
 'https://www.pisos.com/comprar/piso-orcasur28041-62568618264_991741',
 'https://www.pisos.com/comprar/piso-san_sebastian_de_los_reyes_casco_antiguo28701-64253228398_100200',
 'https://www.pisos.com/comprar/piso-prado_de_somosaguas-65865887620_100500',
 'https://www.pisos.com/comprar/piso-goya28001-65933168094_106100',
 'https://www.pisos.com/comprar/piso-acacias-65893764205_106100',
 'https://www.pisos.com/comprar/piso-acacias28005-550

In [11]:
clases:list[str]=list(map(lambda x:str(x),np.unique(list(map(lambda x:x[x[:x.find('-')].rfind('/')+1:x.find('-')],ventas_ads)))))
clases

['apartamento',
 'atico',
 'casa',
 'casa_adosada',
 'casa_pareada',
 'casa_rustica',
 'casa_unifamiliar',
 'chalet',
 'chalet_adosado',
 'chalet_pareado',
 'chalet_rustico',
 'chalet_unifamiliar',
 'duplex',
 'estudio',
 'finca_rustica',
 'loft',
 'piso']

In [27]:
async def consultar_html_ad(url:str,client:httpx.AsyncClient)->str:
    async with SEMAFORO:
        response:httpx.Response=await client.get(url,timeout=10)
        # features:bs4.ResultSet=bs4.BeautifulSoup(response.text,'html.parser').find_all(class_='features__label')
        # return [feature.text for feature in features]
        return response.text
async def consultar_htmls_ads(urls:list[str],comunes:bool=False)->list[str]:
    async with httpx.AsyncClient(transport=httpx.AsyncHTTPTransport(retries=1),limits=httpx.Limits(max_connections=20,max_keepalive_connections=20)) as client:
        tareas:list[Coroutine[Any,Any,str]]=[consultar_html_ad(vivienda,client) for vivienda in urls]
        tareas_completas:list[str]=await asyncio.gather(*tareas)
        return tareas_completas

In [28]:
htmls_ads:list[str]=await consultar_htmls_ads(ventas_ads)

In [32]:
ventas_ads[0].split('/')[-1]

''

In [35]:
for i,add in enumerate(ventas_ads):
    with open(f'./htmls/{add.split('/')[-2]}.html','w',encoding='utf-8') as f:
        f.write(htmls_ads[i])